# TSFaaS — Household Consumption Forecast Example

This notebook walks through a full **train → forecast** round-trip against the TSFaaS HTTP API. We generate a small synthetic hourly consumption profile for one household, train a forecasting model via the service, then request a 24-hour-ahead forecast and plot the result.

## Prerequisites

The notebook talks to the service over HTTP, so **the service must be running before you execute any cells**.

1. Install Docker and Docker Compose.
2. From the repository root, start the stack:

   ```bash
   docker compose up --build
   ```

   This starts three containers: `tsfaas-api` (exposed on port **8001**), `tsfaas-worker`, and `tsfaas-redis`.

3. Wait until `curl http://localhost:8001/health` returns `{"status":"ok"}`. The first build may take several minutes because AutoGluon and its dependencies are large.

### Local Python environment

The notebook itself only needs a few lightweight libraries (the heavy ML stack lives inside the containers):

```bash
pip install requests pandas numpy matplotlib
```

### Timing expectations

Training with the `fast_training` preset on this tiny dataset typically takes **3–10 minutes** on a laptop CPU. The first forecast call after training is fast (a few seconds).

## 1. Imports and configuration

In [ ]:
import json
import time
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

API = "http://localhost:8001"
MODEL_ID = "household-demo"
MODEL_VERSION = "1.0.0"

In [ ]:
# Confirm the service is reachable before going further.
response = requests.get(f"{API}/health", timeout=5)
response.raise_for_status()
print(response.json())

## 2. Generate synthetic household consumption data

We build 14 days of hourly consumption in Wh with a realistic daily shape:
base load + morning peak around 08:00 + evening peak around 20:00 + weekend boost + Gaussian noise.

In [ ]:
rng = np.random.default_rng(42)
start = datetime(2024, 1, 1, tzinfo=timezone.utc)
n_hours = 14 * 24  # 14 days
timestamps = [start + timedelta(hours=i) for i in range(n_hours)]

df = pd.DataFrame({"time": timestamps})
df["hour"] = df["time"].apply(lambda t: t.hour)
df["dayofweek"] = df["time"].apply(lambda t: t.weekday())

morning_peak = 1500 * np.exp(-((df["hour"] - 8) ** 2) / 4)
evening_peak = 2500 * np.exp(-((df["hour"] - 20) ** 2) / 8)
base_load = 200 + 100 * np.sin(2 * np.pi * df["hour"] / 24)
weekend_boost = np.where(df["dayofweek"] >= 5, 300, 0)
noise = rng.normal(0, 50, n_hours)

df["consumption_wh"] = np.clip(
    base_load + morning_peak + evening_peak + weekend_boost + noise,
    50,
    None,
)
print(f"Generated {len(df)} hourly samples")
df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["time"], df["consumption_wh"])
plt.ylabel("Consumption (Wh)")
plt.title("Synthetic household consumption — 14 days")
plt.grid(True, alpha=0.3)
plt.show()

## 3. Register the model

`POST /api/v1/models` writes a metadata record. Training is a separate call.

In [ ]:
response = requests.post(
    f"{API}/api/v1/models",
    json={
        "model_id": MODEL_ID,
        "description": "Demo: household hourly consumption",
        "target": "consumption_wh",
        "freq": "H",
    },
)
print(response.status_code, response.json())

## 4. Helper: poll async jobs

The Compose stack runs with `JOBS_BACKEND=redis`, so `/train` and `/forecast` return immediately with a `job_id`. We poll `/api/v1/jobs/{job_id}` until the job reaches a terminal status.

Status is normalized across backends to `queued` / `running` / `succeeded` / `failed`; the result payload is returned under `detail.result`.

In [ ]:
def wait_for_job(job_id: str, poll_interval: float = 5.0, max_wait: float = 1800.0) -> dict:
    """Poll /jobs/{id} until the job succeeds, fails, or max_wait is exceeded."""
    deadline = time.monotonic() + max_wait
    last_status = None
    while True:
        r = requests.get(f"{API}/api/v1/jobs/{job_id}")
        r.raise_for_status()
        body = r.json()
        status = body.get("status")
        if status != last_status:
            print(f"  status={status}")
            last_status = status
        if status == "succeeded":
            return body
        if status == "failed":
            raise RuntimeError(f"Job failed: {body.get('detail')}")
        if time.monotonic() > deadline:
            raise TimeoutError(f"Job {job_id} did not finish within {max_wait}s")
        time.sleep(poll_interval)

## 5. Train

We keep the example intentionally small:
- `model_preset = "fast_training"` — the fastest AutoGluon preset.
- `lookback_steps = 48` — two days of context.
- `horizon_hours = 24` — predict one day ahead.

The full training payload is a `ForecastTrainingInput` from the `u2` core library.

In [ ]:
training_values = [
    {"time": row["time"].isoformat(), "value": float(row["consumption_wh"])}
    for _, row in df.iterrows()
]

training_payload = {
    "forecast_training_input": {
        "forecast_type": "deterministic",
        "time_parameters": {
            "timestep_minutes": 60,
            "horizon_hours": 24,
        },
        "model_config": {
            "model_id": MODEL_ID,
            "model_preset": "fast_training",
            "model_version": MODEL_VERSION,
            "lookback_steps": 48,
            "eval_metric": "MAE",
        },
        "target_series": {
            "series_id": "house-1",
            "variable_name": "consumption",
            "asset_id": "house-1",
            "asset_type": "household",
            "unit": "Wh",
            "training_values": training_values,
        },
    }
}

print("Submitting training request...")
r = requests.post(f"{API}/api/v1/models/{MODEL_ID}/train", json=training_payload)
r.raise_for_status()
train_job = r.json()
print(train_job)

print("\nWaiting for training to finish (several minutes)...")
train_result = wait_for_job(train_job["job_id"])
print("\nTraining result:")
print(json.dumps(train_result["detail"]["result"], indent=2))

In [ ]:
# Confirm the new version was recorded in the metadata store.
r = requests.get(f"{API}/api/v1/models/{MODEL_ID}")
r.raise_for_status()
print(json.dumps(r.json(), indent=2))

## 6. Forecast

We feed the last 48 hours of observed consumption back to the model as
context and ask for a 24-hour-ahead forecast.

In [ ]:
last_48 = df.tail(48).reset_index(drop=True)
historical_values = [
    {"time": row["time"].isoformat(), "value": float(row["consumption_wh"])}
    for _, row in last_48.iterrows()
]

forecast_payload = {
    "version": "latest",
    "forecast_input": {
        "forecast_type": "deterministic",
        "time_parameters": {
            "timestep_minutes": 60,
            "horizon_hours": 24,
        },
        "target_series": {
            "series_id": "house-1",
            "variable_name": "consumption",
            "asset_id": "house-1",
            "asset_type": "household",
            "unit": "Wh",
            "historical_values": historical_values,
        },
    },
}

r = requests.post(f"{API}/api/v1/models/{MODEL_ID}/forecast", json=forecast_payload)
r.raise_for_status()
forecast_job = r.json()
print(forecast_job)

print("\nWaiting for forecast...")
forecast_result = wait_for_job(forecast_job["job_id"])
forecast = forecast_result["detail"]["result"]
print(f"\nReceived {len(forecast['forecast_series']['values'])} forecasted points")

## 7. Plot the forecast

We overlay the 48-hour context window and the 24-hour prediction on a single chart.

In [ ]:
forecast_df = pd.DataFrame(
    [
        {"time": pd.to_datetime(v["time"]), "value": v["value"]}
        for v in forecast["forecast_series"]["values"]
    ]
)

plt.figure(figsize=(12, 4))
plt.plot(last_48["time"], last_48["consumption_wh"], label="history (last 48h)")
plt.plot(
    forecast_df["time"],
    forecast_df["value"],
    label="forecast (next 24h)",
    color="tab:orange",
    marker="o",
    markersize=4,
)
plt.axvline(last_48["time"].iloc[-1], color="gray", linestyle=":", label="now")
plt.ylabel("Consumption (Wh)")
plt.legend()
plt.title(f"Household forecast — model={MODEL_ID}, version={MODEL_VERSION}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Alternative sources: URL data + remote model store

The steps above used **inline** data (`forecast_training_input` / `forecast_input`) and the default
**local volume** model store. The same endpoints accept a `data_source` to fetch input from a URL, and a
`model_source` to persist/load the artifact from a remote HTTP(S) or SFTP store. The two axes are
independent — you can mix inline/URL data with volume/URL models freely.

See [`doc/sources.md`](sources.md) for the full field reference. The cells below build the exact request
payloads for these categories. They are **illustrative**: running them needs a reachable data endpoint and
model store, so the network calls are guarded behind `RUN_URL_EXAMPLES`.

In [ ]:
RUN_URL_EXAMPLES = False  # flip to True once you have a reachable data endpoint + model store

# Train from URL data, and push the trained artifact to a remote HTTPS store.
# The Bearer token is read from the worker's environment (never sent in the body).
url_training_payload = {
    "data_source": {
        "type": "url",
        "request": {
            "method": "GET",
            "url": "https://data.example.org/households/house-1/training.json",
            "headers": {},
            "query": {"from": "2024-01-01", "to": "2024-01-14"},
            "timeout_seconds": 30,
            "retry": {"max_attempts": 3, "backoff_seconds": 1.0},
        },
    },
    "model_source": {
        "type": "url",
        "location": {
            "scheme": "https",
            "url": "https://models.example.org/house-1/1.0.0.tar.gz",
            "upload_method": "PUT",
            "auth": {"token": {"env": "MODEL_STORE_TOKEN"}},
        },
    },
}
print(json.dumps(url_training_payload, indent=2))

if RUN_URL_EXAMPLES:
    r = requests.post(f"{API}/api/v1/models/{MODEL_ID}/train", json=url_training_payload)
    r.raise_for_status()
    print(wait_for_job(r.json()["job_id"]))

In [ ]:
# Forecast by loading the artifact from the remote store, with input data also
# fetched from a URL. Omitting `model_source` would instead load from the local volume.
url_forecast_payload = {
    "version": "latest",
    "data_source": {
        "type": "url",
        "request": {
            "method": "GET",
            "url": "https://data.example.org/households/house-1/recent.json",
            "timeout_seconds": 30,
        },
    },
    "model_source": {
        "type": "url",
        "location": {
            "scheme": "https",
            "url": "https://models.example.org/house-1/1.0.0.tar.gz",
            "auth": {"token": {"env": "MODEL_STORE_TOKEN"}},
        },
    },
}
print(json.dumps(url_forecast_payload, indent=2))

if RUN_URL_EXAMPLES:
    r = requests.post(f"{API}/api/v1/models/{MODEL_ID}/forecast", json=url_forecast_payload)
    r.raise_for_status()
    print(wait_for_job(r.json()["job_id"]))